# Analyse et reporting SQL

Layla EL GOZMIR · Projet portfolio

Ce notebook commente les résultats du pipeline. Le code d'analyse versionné reste la source de vérité.
Les résultats fournis sont issus d'une exécution réelle, et non de valeurs saisies manuellement.
Exécuter toutes les cellules ; mettre `REBUILD=True` pour recalculer (Internet requis au premier lancement).


In [1]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, Markdown, Image
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
REBUILD = False


In [2]:
from run import run
if REBUILD or not (ROOT / 'reports/metrics.json').exists():
    result = run()
else:
    result = json.loads((ROOT / 'reports/metrics.json').read_text(encoding='utf-8'))
display(Markdown((ROOT / 'reports/synthese.md').read_text(encoding='utf-8')))


# Résultats exécutés

412 factures, 59 acheteurs, CA de 2328.60 unités monétaires.
Période : 2009-01-01 00:00:00 à 2013-12-22 00:00:00.
Premier pays de facturation : USA (22.46 % du CA).
Premier genre : Rock (826.65).

Les contrôles de jointure, de clés étrangères et de rapprochement des montants passent.
Chinook contient des ventes générées : ces résultats illustrent la méthode et ne décrivent pas un marché réel.
Le taux de réachat porte sur toute la fenêtre observée, sans interprétation causale ni prévision.
Une décision commerciale nécessiterait des données réelles sur la marge, les coûts et les périodes comparables.


## Des questions métier aux requêtes

Le grain facture sert au panier moyen. Le grain ligne sert à répartir les ventes par genre.
La croissance compare deux mois consécutifs avec LAG ; le premier mois reste indéfini.


In [3]:
display(pd.DataFrame([result['totals']]))
display(result['checks'])
display(pd.read_csv(ROOT / 'reports/monthly_growth.csv').head(12))

,orders,revenue_cents,buyers,first_date,last_date
0,412,232860,59,2009-01-01 00:00:00,2013-12-22 00:00:00


{'foreign_key_errors': 0,
 'invoice_mismatches': 0,
 'joined_line_count_matches': True,
 'revenue_totals_match': True}

,month,orders,active_customers,revenue_cents,average_order_cents,previous_cents,growth_percent
0,2009-01,6,6,3564,594.000000,NaN,NaN
1,2009-02,7,7,3762,537.428571,3564.0,5.56
2,2009-03,7,7,3762,537.428571,3762.0,0.00
3,2009-04,7,7,3762,537.428571,3762.0,0.00
4,2009-05,7,7,3762,537.428571,3762.0,0.00
5,2009-06,7,7,3762,537.428571,3762.0,0.00
6,2009-07,7,7,3762,537.428571,3762.0,0.00
7,2009-08,7,7,3762,537.428571,3762.0,0.00
8,2009-09,7,7,3762,537.428571,3762.0,0.00
9,2009-10,7,7,3762,537.428571,3762.0,0.00


In [4]:
display(pd.read_csv(ROOT / 'reports/countries.csv').head(10))
display(pd.read_csv(ROOT / 'reports/genres.csv').head(10))

,billing_country,orders,buyers,revenue_cents,share_percent
0,USA,91,13,52306,22.46
1,Canada,56,8,30396,13.05
2,France,35,5,19510,8.38
3,Brazil,35,5,19010,8.16
4,Germany,28,4,15648,6.72
5,United Kingdom,21,3,11286,4.85
6,Czech Republic,14,2,9024,3.88
7,Portugal,14,2,7724,3.32
8,India,13,2,7526,3.23
9,Chile,7,1,4662,2.00


,genre,units,revenue_cents,revenue_rank
0,Rock,835,82665,1
1,Latin,386,38214,2
2,Metal,264,26136,3
3,Alternative & Punk,244,24156,4
4,TV Shows,47,9353,5
5,Jazz,80,7920,6
6,Blues,61,6039,7
7,Drama,29,5771,8
8,Classical,41,4059,9
9,R&B/Soul,41,4059,9


## Contrôle central

Le rapprochement est vide si chaque total facture égale la somme de ses lignes. Les ventes de Chinook sont synthétiques : les constats ne peuvent pas devenir des recommandations réelles de marché.


In [5]:
assert pd.read_csv(ROOT / 'reports/reconciliation.csv').empty
print('Rapprochement validé')

Rapprochement validé


## À expliquer en entretien

1. Quelle est l'unité d'observation et comment les doublons sont-ils traités ?
2. Quelles informations servent à sélectionner les paramètres et lesquelles servent à évaluer ?
3. Quelle décision métier est envisageable, et quelles limites empêchent une conclusion plus forte ?

Voir le README pour les sources, les limites, les commandes et l'attribution de l'assistance IA.
